# M1 Shared-Reference Mitigation Pipeline — BDHS 2022 Child Stunting

**Purpose:** compare a conventional, method-specific reference setup (**Baseline**) with **M1**, which gives SHAP, LIME, and Integrated Gradients the same *observed training-reference rows* within each fold. Measure explanation disagreement, repeat/reference instability, and a realistic conditional feature-replacement faithfulness check.

## Important before running
- The notebook uses `hw70 < -200` for stunting and group-aware 5-fold CV by `v001` (PSU/cluster), after restricting `hw70` to the plausible DHS z-score range `[-600, 600]`.
- **Default feature mode is a 4-feature smoke test** (`v729`, `b19`, `v191a`, `m18`) from the previously discussed pilot. It is **not Arm A**. For the thesis run, set `FEATURE_SET_MODE = "ARM_A_14"` and paste the exact 14-feature list from your approved v3.1.1 protocol/config into `ARM_A_FEATURES`. This notebook intentionally does not guess the missing 14-variable list.
- No variable is treated as a cause of stunting. Explanations describe the fitted model, not causal effects or clinical risk.
- M1 is an experimental mitigation, not a guaranteed improvement. The outputs must show whether it works.
- LIME still generates local synthetic samples internally; its categorical variables are explicitly marked as categorical. The **faithfulness check** separately uses nearest-neighbour donor values from observed training records and changes only one feature at a time.

## Experimental flow
1. Load the BDHS 2022 Kids Recode file and create the stunting target.
2. Use 5-fold `StratifiedGroupKFold`; keep entire clusters in either train or test.
3. Fit Logistic Regression, Random Forest, and MLP within each fold.
4. Select evaluation children from the held-out fold and create reference rows using the training fold only.
5. Explain with SHAP and LIME for all models; IG for MLP only.
6. Compare Baseline (independent reference banks of equal size) against M1 (same observed reference rows for all explainers/models within the fold).
7. Calibrate explanation instability on a small repeated subset.
8. Check faithfulness by replacing one feature with an observed value from a nearest training donor matched on the other features.
9. Save all tables and figures to the Thesis results folder.

In [ ]:
# Google Colab setup: install missing explanation libraries.
%pip -q install shap lime captum

In [ ]:
# Mount Google Drive (skip this cell only if you will upload the .DTA file directly).
from google.colab import drive
drive.mount('/content/drive')

## 1. Configuration
Edit this cell first. If the BDHS codebook identifies special missing codes for any selected predictor, add them to `FEATURE_MISSING_CODES`. For the real thesis run, use `FEATURE_SET_MODE = "ARM_A_14"` and paste your exact 14 features. Start with `RUN_MODE = "pilot"` to check that everything runs; use `"full"` for 40 held-out children per fold after the pilot completes cleanly.

In [ ]:
from pathlib import Path
import os, json, time, warnings, itertools, math
from datetime import datetime

# Data location. Leave empty to let the notebook search common Colab/Drive locations,
# then offer an upload prompt if it cannot find a suitable dataset.
DATA_PATH = ""  # e.g. /content/drive/MyDrive/Thesis/data/BDKR81FL.DTA
PROJECT_ROOT = "/content/drive/MyDrive/Thesis"
RESULTS_ROOT = os.path.join(PROJECT_ROOT, "results")

# FEATURE_SET_MODE options: "DEBUG_4" or "ARM_A_14".
FEATURE_SET_MODE = "DEBUG_4"
DEBUG_FEATURES = ["v729", "b19", "v191a", "m18"]  # smoke-test only; not the thesis Arm A
ARM_A_FEATURES = []  # REQUIRED: paste the exact 14 predictor names from your approved v3.1.1 protocol

# Optional overrides if automatic type inference is wrong for any DHS variable.
# Categorical DHS codes should be listed as categorical; continuous measures as continuous.
CATEGORICAL_FEATURES_OVERRIDE = []
CONTINUOUS_FEATURES_OVERRIDE = []

# Optional feature-specific special-missing codes, verified against the BDHS recode codebook.
# Example format only: {"m18": [9, 99]}. Do NOT add codes without checking each variable.
FEATURE_MISSING_CODES = {}

TARGET_COL = "hw70"
CLUSTER_COL = "v001"
TARGET_VALID_MIN = -600
TARGET_VALID_MAX = 600
STUNTING_CUTOFF = -200  # hw70 is stored in tenths of a z-score; < -200 means HAZ < -2 SD

RUN_MODE = "pilot"  # "pilot" = 10 explained cases/fold; "full" = 40/fold
N_SPLITS = 5
SEED = 31415
N_EXPLAIN_PER_FOLD = {"pilot": 10, "full": 40}[RUN_MODE]
N_REFERENCE_ROWS = 30
SHAP_NSAMPLES = 100
LIME_NUM_SAMPLES = 1500
IG_STEPS = 24
NOISE_CASES_PER_FOLD = 5  # first few explained cases/fold get repeated runs
NOISE_REPEATS_TOTAL = 3  # includes the main explanation; 3 means main + 2 repeats
TOP_K = 5
MAX_AUTO_CATEGORICAL_LEVELS = 16

# Avoid exporting raw child/household identifiers. Only local within-fold case numbers are saved.
RANDOM_STATE = SEED
print("Feature mode:", FEATURE_SET_MODE, "| Run mode:", RUN_MODE, "| Cases/fold:", N_EXPLAIN_PER_FOLD)
if FEATURE_SET_MODE == "DEBUG_4":
    FEATURES = DEBUG_FEATURES.copy()
    FEATURE_SET_LABEL = "debug_four_feature_smoke_test_NOT_ARM_A"
elif FEATURE_SET_MODE == "ARM_A_14":
    FEATURES = [str(x).lower() for x in ARM_A_FEATURES]
    if len(FEATURES) != 14 or len(set(FEATURES)) != 14:
        raise ValueError("ARM_A_14 selected, but ARM_A_FEATURES must contain the exact 14 distinct variables from your protocol.")
    FEATURE_SET_LABEL = "arm_a_14"
else:
    raise ValueError("FEATURE_SET_MODE must be 'DEBUG_4' or 'ARM_A_14'.")
if set(CATEGORICAL_FEATURES_OVERRIDE) & set(CONTINUOUS_FEATURES_OVERRIDE):
    raise ValueError("A feature cannot be both categorical and continuous in the override lists.")

## 2. Imports and data loading
The loader prefers a Stata `.DTA` file. If a different file is selected, check that it contains the raw DHS recode variables rather than a previously transformed feature matrix.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.exceptions import ConvergenceWarning
import shap
from lime.lime_tabular import LimeTabularExplainer
import torch
import torch.nn as nn
import torch.nn.functional as F
from captum.attr import IntegratedGradients

warnings.filterwarnings("ignore", category=ConvergenceWarning)
np.random.seed(SEED)
torch.manual_seed(SEED)


def _candidate_data_files():
    candidates = []
    roots = [Path("/content"), Path(PROJECT_ROOT), Path(PROJECT_ROOT) / "data"]
    extensions = {".dta", ".csv", ".parquet", ".sav"}
    for root in roots:
        if root.exists():
            try:
                for p in root.iterdir():
                    if p.is_file() and p.suffix.lower() in extensions:
                        candidates.append(p)
            except PermissionError:
                pass
    # Prefer the raw Stata Kids Recode if present.
    kids = [p for p in candidates if p.name.lower().startswith("bdkr81") and p.suffix.lower() == ".dta"]
    if len(kids) == 1:
        return str(kids[0])
    stata = [p for p in candidates if p.suffix.lower() == ".dta"]
    if len(stata) == 1:
        return str(stata[0])
    return None


def resolve_data_path():
    if DATA_PATH.strip():
        if not os.path.isfile(DATA_PATH):
            raise FileNotFoundError(f"DATA_PATH does not exist: {DATA_PATH}")
        return DATA_PATH
    auto = _candidate_data_files()
    if auto:
        print("Auto-selected:", auto)
        return auto
    try:
        from google.colab import files
        print("Could not uniquely find the dataset. Upload the raw BDHS Kids Recode .DTA file now.")
        uploaded = files.upload()
        if not uploaded:
            raise FileNotFoundError("No dataset uploaded. Set DATA_PATH manually and rerun.")
        return next(iter(uploaded.keys()))
    except ImportError as e:
        raise FileNotFoundError("Set DATA_PATH to your raw BDHS file.") from e


def load_dataset(path):
    suffix = Path(path).suffix.lower()
    if suffix == ".dta":
        return pd.read_stata(path, convert_categoricals=False)
    if suffix == ".csv":
        return pd.read_csv(path, low_memory=False)
    if suffix == ".parquet":
        return pd.read_parquet(path)
    if suffix == ".sav":
        return pd.read_spss(path, convert_categoricals=False)
    raise ValueError(f"Unsupported file extension: {suffix}")

DATA_PATH_RESOLVED = resolve_data_path()
df = load_dataset(DATA_PATH_RESOLVED)
df.columns = [str(c).strip().lower() for c in df.columns]
print("Loaded:", DATA_PATH_RESOLVED)
print("Shape:", df.shape)
print("First columns:", df.columns[:20].tolist())

required = {TARGET_COL, CLUSTER_COL, *FEATURES}
missing_required = sorted(required - set(df.columns))
if missing_required:
    raise KeyError(f"Required columns missing: {missing_required}. Check DATA_PATH and feature config.")

# Target validity: official DHS anthropometry special-missing codes fall outside plausible HAZ range.
hw70 = pd.to_numeric(df[TARGET_COL], errors="coerce")
valid_target = hw70.between(TARGET_VALID_MIN, TARGET_VALID_MAX, inclusive="both")
cluster_numeric = pd.to_numeric(df[CLUSTER_COL], errors="coerce")
valid_cluster = cluster_numeric.notna()
analytic = df.loc[valid_target & valid_cluster, [*FEATURES, TARGET_COL, CLUSTER_COL]].copy().reset_index(drop=True)
analytic[TARGET_COL] = pd.to_numeric(analytic[TARGET_COL], errors="coerce")
analytic[CLUSTER_COL] = pd.to_numeric(analytic[CLUSTER_COL], errors="coerce")
y = (analytic[TARGET_COL] < STUNTING_CUTOFF).astype(int).to_numpy()
groups = analytic[CLUSTER_COL].to_numpy()

# Convert candidate features to numeric; raw DHS recode predictor values should be numeric codes/measures.
for f in FEATURES:
    analytic[f] = pd.to_numeric(analytic[f], errors="coerce")
for f, codes in FEATURE_MISSING_CODES.items():
    f = str(f).lower()
    if f not in FEATURES:
        raise ValueError(f"FEATURE_MISSING_CODES contains {f}, which is not in FEATURES.")
    analytic.loc[analytic[f].isin(list(codes)), f] = np.nan

print(f"Analytic rows with valid hw70 and cluster: {len(analytic):,}")
print(f"Unique clusters: {pd.Series(groups).nunique():,}")
print("Stunting prevalence (unweighted):", round(float(y.mean()), 4))
print(pd.Series(y).value_counts().rename(index={0:"not_stunted", 1:"stunted"}))
if pd.Series(y).nunique() != 2:
    raise ValueError("The valid analytic sample must contain both stunting classes.")
if CLUSTER_COL in FEATURES or TARGET_COL in FEATURES:
    raise ValueError("Do not include cluster ID or outcome itself as a predictor.")

## 3. Preprocessing and model helpers
Feature type detection is performed using each fold's training data only. Small-integer DHS codes are treated as categorical by default; override the type lists in Configuration if necessary. Imputation statistics are also learned only from the training fold.

In [ ]:
def infer_feature_types(X_train, features):
    categorical, continuous = [], []
    cat_override = {str(x).lower() for x in CATEGORICAL_FEATURES_OVERRIDE}
    cont_override = {str(x).lower() for x in CONTINUOUS_FEATURES_OVERRIDE}
    for f in features:
        if f in cat_override:
            categorical.append(f); continue
        if f in cont_override:
            continuous.append(f); continue
        s = pd.to_numeric(X_train[f], errors="coerce").dropna()
        if s.empty:
            # A completely missing training-fold feature is treated as numeric and later gets a safe fill.
            continuous.append(f); continue
        integer_like = np.all(np.isclose(s.to_numpy(dtype=float), np.round(s.to_numpy(dtype=float))))
        n_unique = int(s.nunique())
        if integer_like and n_unique <= MAX_AUTO_CATEGORICAL_LEVELS:
            categorical.append(f)
        else:
            continuous.append(f)
    return categorical, continuous


def fit_raw_imputation(X_train, features, categorical, continuous):
    stats = {}
    for f in features:
        s = pd.to_numeric(X_train[f], errors="coerce")
        nonmissing = s.dropna()
        if f in categorical:
            mode = nonmissing.mode(dropna=True)
            stats[f] = float(mode.iloc[0]) if len(mode) else 0.0
        else:
            stats[f] = float(nonmissing.median()) if len(nonmissing) else 0.0
    return stats


def apply_raw_imputation(X, features, stats):
    out = pd.DataFrame(index=X.index)
    for f in features:
        s = pd.to_numeric(X[f], errors="coerce")
        out[f] = s.fillna(stats[f]).astype(float)
    return out[features]


def make_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:  # scikit-learn compatibility
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def make_preprocessor(features, categorical, continuous):
    transformers = []
    if continuous:
        transformers.append(("num", StandardScaler(), continuous))
    if categorical:
        transformers.append(("cat", make_ohe(), categorical))
    return ColumnTransformer(transformers, remainder="drop", sparse_threshold=0.0)


def fit_models(X_train, y_train, categorical, continuous, seed):
    specs = {
        "LogisticRegression": LogisticRegression(max_iter=2000, solver="liblinear", random_state=seed),
        "RandomForest": RandomForestClassifier(n_estimators=250, min_samples_leaf=3, n_jobs=-1, random_state=seed),
        "MLP": MLPClassifier(hidden_layer_sizes=(32, 16), activation="relu", solver="adam",
                             alpha=1e-3, max_iter=350, early_stopping=True,
                             validation_fraction=0.1, random_state=seed),
    }
    fitted = {}
    for name, estimator in specs.items():
        pipe = Pipeline([("prep", make_preprocessor(FEATURES, categorical, continuous)), ("clf", estimator)])
        pipe.fit(X_train, y_train)
        fitted[name] = pipe
    return fitted


def feature_groups_in_transformed_space(prep, features, categorical, continuous):
    groups_out = {}
    out_slices = prep.output_indices_
    if continuous:
        s = out_slices["num"]
        for j, f in enumerate(continuous):
            groups_out[f] = np.array([s.start + j], dtype=int)
    if categorical:
        s = out_slices["cat"]
        ohe = prep.named_transformers_["cat"]
        offset = s.start
        for f, cats in zip(categorical, ohe.categories_):
            width = len(cats)
            groups_out[f] = np.arange(offset, offset + width, dtype=int)
            offset += width
    # Preserve original feature order for aggregation/output.
    return {f: groups_out[f] for f in features}


def predict_positive(model, values):
    arr = np.asarray(values, dtype=float)
    if arr.ndim == 1:
        arr = arr.reshape(1, -1)
    frame = pd.DataFrame(arr, columns=FEATURES)
    return model.predict_proba(frame)[:, 1]


def sample_rows(X, n, seed):
    n = min(int(n), len(X))
    if n <= 0:
        raise ValueError("Cannot sample a reference bank from an empty training fold.")
    return X.sample(n=n, replace=False, random_state=seed).reset_index(drop=True)


def make_reference_banks(X_train, fold):
    # Baseline: independent, equal-size observed reference banks for each explainer.
    # M1: one observed reference bank shared by all explainers and model types in this fold.
    n_ref = min(N_REFERENCE_ROWS, len(X_train))
    baseline = {
        "SHAP": sample_rows(X_train, n_ref, SEED + fold * 101 + 11),
        "LIME": sample_rows(X_train, n_ref, SEED + fold * 101 + 23),
        "IG": sample_rows(X_train, n_ref, SEED + fold * 101 + 37),
    }
    shared = sample_rows(X_train, n_ref, SEED + fold * 101 + 53)
    return {"Baseline": baseline, "M1_shared_reference": {"SHAP": shared, "LIME": shared, "IG": shared}}

## 4. Explanation functions
- **SHAP:** model-agnostic Kernel SHAP on the raw predictors. It replaces masked values with observed reference-row values and returns one attribution per original DHS feature.
- **LIME:** the explainer receives the same reference bank; categorical codes are declared categorical, and local samples are generated using LIME's tabular mechanism.
- **IG:** used only for the MLP. Its baseline is a set of observed training rows transformed through the fold's fitted preprocessing. IG values over one-hot columns are summed back to the original DHS feature.

In [ ]:
def build_shap_explainer(model, background, seed):
    # predict_positive returns a one-dimensional probability array, so Kernel SHAP produces one value per raw feature.
    return shap.KernelExplainer(lambda a: predict_positive(model, a), background[FEATURES].to_numpy(dtype=float))


def explain_shap(explainer, row, seed):
    vals = explainer.shap_values(row[FEATURES].to_numpy(dtype=float).reshape(1, -1),
                                 nsamples=max(SHAP_NSAMPLES, 2 * len(FEATURES) + 2),
                                 l1_reg=0.0, silent=True, rseed=int(seed))
    if isinstance(vals, list):
        vals = vals[-1]
    vals = np.asarray(vals)
    if vals.ndim == 3:
        vals = vals[0, :, -1]
    elif vals.ndim == 2:
        vals = vals[0]
    else:
        vals = vals.reshape(-1)
    if len(vals) != len(FEATURES):
        raise ValueError(f"Unexpected SHAP shape {np.asarray(vals).shape}; expected {len(FEATURES)} feature attributions.")
    return vals.astype(float)


def explain_lime(model, row, background, categorical, seed):
    cat_indices = [FEATURES.index(f) for f in categorical]
    explainer = LimeTabularExplainer(
        training_data=background[FEATURES].to_numpy(dtype=float),
        feature_names=FEATURES,
        categorical_features=cat_indices,
        mode="classification",
        class_names=["not_stunted", "stunted"],
        discretize_continuous=True,
        feature_selection="none",
        random_state=int(seed),
    )
    def predict_two_classes(a):
        p = predict_positive(model, a)
        return np.column_stack([1.0 - p, p])
    exp = explainer.explain_instance(row[FEATURES].to_numpy(dtype=float), predict_two_classes,
                                     labels=(1,), num_features=len(FEATURES), num_samples=LIME_NUM_SAMPLES)
    out = np.zeros(len(FEATURES), dtype=float)
    for feature_idx, weight in exp.as_map().get(1, []):
        if 0 <= int(feature_idx) < len(FEATURES):
            out[int(feature_idx)] = float(weight)
    return out


class TorchMirrorMLP(nn.Module):
    """Torch mirror of a fitted binary sklearn MLPClassifier (ReLU hidden layers, sigmoid output)."""
    def __init__(self, sklearn_mlp):
        super().__init__()
        self.layers = nn.ModuleList()
        for W, b in zip(sklearn_mlp.coefs_, sklearn_mlp.intercepts_):
            layer = nn.Linear(W.shape[0], W.shape[1])
            with torch.no_grad():
                layer.weight.copy_(torch.tensor(W.T, dtype=torch.float32))
                layer.bias.copy_(torch.tensor(b, dtype=torch.float32))
            self.layers.append(layer)
        self.eval()

    def forward(self, x):
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if i < len(self.layers) - 1:
                x = F.relu(x)
        return torch.sigmoid(x.squeeze(-1))


def make_torch_mlp_mirror(model):
    clf = model.named_steps["clf"]
    torch_model = TorchMirrorMLP(clf)
    return torch_model


def explain_ig(model, row, background, feature_groups, torch_model):
    prep = model.named_steps["prep"]
    xz = np.asarray(prep.transform(row[FEATURES].to_frame().T), dtype=np.float32)
    bz = np.asarray(prep.transform(background[FEATURES]), dtype=np.float32)
    if len(bz) == 0:
        raise ValueError("IG requires at least one observed reference row.")
    inputs = torch.tensor(np.repeat(xz, len(bz), axis=0), dtype=torch.float32)
    baselines = torch.tensor(bz, dtype=torch.float32)
    ig = IntegratedGradients(torch_model)
    attrs = ig.attribute(inputs, baselines=baselines, n_steps=IG_STEPS, internal_batch_size=64)
    enc_attr = attrs.detach().cpu().numpy().mean(axis=0)
    out = []
    for f in FEATURES:
        out.append(float(enc_attr[feature_groups[f]].sum()))
    return np.asarray(out, dtype=float)


def spearman_safe(a, b):
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    if len(a) < 2 or np.allclose(a, a[0]) or np.allclose(b, b[0]):
        return np.nan
    return float(spearmanr(a, b).statistic)


def compare_importance(a, b, k):
    # Rankings use absolute attribution magnitude; sign/direction is available in the raw output table.
    ia, ib = np.abs(np.asarray(a)), np.abs(np.asarray(b))
    rho = spearman_safe(ia, ib)
    rank_disagreement = float(1.0 - (rho + 1.0) / 2.0) if np.isfinite(rho) else np.nan
    k = max(1, min(int(k), len(ia)))
    ta = set(np.argsort(-ia)[:k].tolist())
    tb = set(np.argsort(-ib)[:k].tolist())
    jaccard_distance = 1.0 - len(ta & tb) / max(1, len(ta | tb))
    return rho, rank_disagreement, float(jaccard_distance)

## 5. Cross-validation experiment: Baseline vs M1
`pilot` explains 10 cases per fold; `full` explains 40 per fold. References are sampled **only from the current fold's training data**. The same held-out cases are explained under both modes. IG is run only on MLP, since the other model types do not expose the differentiable input pathway needed here.

In [ ]:
cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
split_list = list(cv.split(analytic[FEATURES], y, groups=groups))

fold_metric_rows = []
explanation_rows = []
main_attributions = {}  # (fold, case_no, model_name, mode, method) -> attribution vector
noise_attributions = {}  # same key -> list of repeat vectors; only for selected noise-calibration cases
case_context = {}  # (fold, case_no) -> data used later for the conditional faithfulness test
fold_contexts = {}

for fold_idx, (train_idx, test_idx) in enumerate(split_list, start=1):
    print(f"\n========== FOLD {fold_idx}/{N_SPLITS} ==========")
    X_train_raw0 = analytic.iloc[train_idx][FEATURES].copy()
    X_test_raw0 = analytic.iloc[test_idx][FEATURES].copy()
    y_train = y[train_idx]
    y_test = y[test_idx]

    categorical, continuous = infer_feature_types(X_train_raw0, FEATURES)
    imputation_stats = fit_raw_imputation(X_train_raw0, FEATURES, categorical, continuous)
    X_train = apply_raw_imputation(X_train_raw0, FEATURES, imputation_stats).reset_index(drop=True)
    X_test_all = apply_raw_imputation(X_test_raw0, FEATURES, imputation_stats).reset_index(drop=True)
    fitted_models = fit_models(X_train, y_train, categorical, continuous, SEED + fold_idx)
    feature_groups_by_model = {}
    torch_models = {}

    for model_name, model in fitted_models.items():
        p_test = model.predict_proba(X_test_all)[:, 1]
        fold_metric_rows.append({
            "fold": fold_idx, "model": model_name,
            "n_train": len(train_idx), "n_test": len(test_idx),
            "n_train_clusters": int(pd.Series(groups[train_idx]).nunique()),
            "n_test_clusters": int(pd.Series(groups[test_idx]).nunique()),
            "auroc": float(roc_auc_score(y_test, p_test)) if len(np.unique(y_test)) == 2 else np.nan,
            "average_precision": float(average_precision_score(y_test, p_test)),
            "brier_score": float(brier_score_loss(y_test, p_test)),
        })
        feature_groups_by_model[model_name] = feature_groups_in_transformed_space(
            model.named_steps["prep"], FEATURES, categorical, continuous)
        if model_name == "MLP":
            torch_models[model_name] = make_torch_mlp_mirror(model)
            # Sanity-check the PyTorch mirror against sklearn's positive-class probability.
            z_check = np.asarray(model.named_steps["prep"].transform(X_test_all.iloc[:min(5, len(X_test_all))]), dtype=np.float32)
            with torch.no_grad():
                p_torch = torch_models[model_name](torch.tensor(z_check, dtype=torch.float32)).cpu().numpy()
            p_sklearn = model.predict_proba(X_test_all.iloc[:len(z_check)])[:, 1]
            max_err = float(np.max(np.abs(p_torch - p_sklearn)))
            print(f"MLP Torch-mirror max probability error: {max_err:.3g}")
            if max_err > 2e-4:
                raise RuntimeError("Torch mirror differs from the fitted sklearn MLP; stopping to protect IG validity.")

    # Choose a stratified sample of the held-out fold for explanation.
    n_case = min(N_EXPLAIN_PER_FOLD, len(test_idx))
    local_positions = np.arange(len(test_idx))
    if n_case < len(local_positions) and len(np.unique(y_test)) == 2 and min(np.bincount(y_test)) >= 2:
        chosen_local, _ = train_test_split(local_positions, train_size=n_case, stratify=y_test,
                                          random_state=SEED + fold_idx)
    else:
        rng = np.random.default_rng(SEED + fold_idx)
        chosen_local = np.sort(rng.choice(local_positions, size=n_case, replace=False))
    X_eval = X_test_all.iloc[chosen_local].reset_index(drop=True)
    y_eval = y_test[chosen_local]

    ref_banks = make_reference_banks(X_train, fold_idx)
    shap_explainer_cache = {}
    for mode in ["Baseline", "M1_shared_reference"]:
        for model_name, model in fitted_models.items():
            bg_shap = ref_banks[mode]["SHAP"]
            shap_explainer_cache[(mode, model_name)] = build_shap_explainer(
                model, bg_shap, SEED + fold_idx * 100 + (0 if mode == "Baseline" else 1))

    for case_zero, row in X_eval.iterrows():
        case_no = int(case_zero + 1)
        y_true_case = int(y_eval[case_zero])
        case_context[(fold_idx, case_no)] = {
            "row": row.copy(), "y_true": y_true_case,
            "X_train": X_train, "categorical": categorical, "continuous": continuous,
            "fitted_models": fitted_models,
            "feature_groups_by_model": feature_groups_by_model,
            "torch_models": torch_models,
        }
        do_noise = case_no <= NOISE_CASES_PER_FOLD
        for mode in ["Baseline", "M1_shared_reference"]:
            for model_name, model in fitted_models.items():
                methods = ["SHAP", "LIME"] + (["IG"] if model_name == "MLP" else [])
                for method in methods:
                    bg = ref_banks[mode][method]
                    base_seed = SEED + fold_idx * 10000 + case_no * 101 + (0 if mode == "Baseline" else 50000) + (0 if method == "SHAP" else 100 if method == "LIME" else 200)
                    if method == "SHAP":
                        attr = explain_shap(shap_explainer_cache[(mode, model_name)], row, base_seed)
                    elif method == "LIME":
                        attr = explain_lime(model, row, bg, categorical, base_seed)
                    else:
                        attr = explain_ig(model, row, bg, feature_groups_by_model[model_name], torch_models[model_name])
                    key = (fold_idx, case_no, model_name, mode, method)
                    main_attributions[key] = attr
                    explanation_rows.extend([{
                        "fold": fold_idx, "case_no_within_fold": case_no, "y_true": y_true_case,
                        "feature_set": FEATURE_SET_LABEL, "mode": mode, "model": model_name,
                        "method": method, "feature": f, "attribution": float(a),
                        "importance_abs": float(abs(a)),
                    } for f, a in zip(FEATURES, attr)])

                    if do_noise:
                        reps = [attr.copy()]
                        for rep_idx in range(1, NOISE_REPEATS_TOTAL):
                            rep_seed = base_seed + 700001 * rep_idx
                            if method == "SHAP":
                                rep_attr = explain_shap(shap_explainer_cache[(mode, model_name)], row, rep_seed)
                            elif method == "LIME":
                                rep_attr = explain_lime(model, row, bg, categorical, rep_seed)
                            else:
                                # For IG, repeat the reference-row draw to estimate sensitivity to the baseline bank.
                                ig_bg = sample_rows(X_train, min(N_REFERENCE_ROWS, len(X_train)), rep_seed)
                                rep_attr = explain_ig(model, row, ig_bg, feature_groups_by_model[model_name], torch_models[model_name])
                            reps.append(rep_attr.copy())
                        noise_attributions[key] = reps

    fold_contexts[fold_idx] = {
        "X_train": X_train, "X_eval": X_eval, "fitted_models": fitted_models,
        "categorical": categorical, "continuous": continuous,
        "feature_groups_by_model": feature_groups_by_model,
    }
    print("Explained cases:", len(X_eval), "| reference rows:", min(N_REFERENCE_ROWS, len(X_train)),
          "| categorical:", categorical, "| continuous:", continuous)

fold_metrics = pd.DataFrame(fold_metric_rows)
explanations_long = pd.DataFrame(explanation_rows)
print("\nModel performance (fold-level preliminary metrics):")
display(fold_metrics)

## 6. Quantify method disagreement and model-type disagreement
Rank comparisons use **absolute attribution magnitudes** so they compare which features are considered important. Signed attributions are retained in `explanations_long.csv` for separate direction analyses. A top-k Jaccard distance of 0 means identical top features; 1 means no top-feature overlap.

In [ ]:
# Within each model: compare different explainers on the same case.
within_rows = []
contexts = sorted({(k[0], k[1], k[2], k[3]) for k in main_attributions})
for fold_idx, case_no, model_name, mode in contexts:
    methods_here = sorted([k[4] for k in main_attributions if k[:4] == (fold_idx, case_no, model_name, mode)])
    for m1, m2 in itertools.combinations(methods_here, 2):
        a = main_attributions[(fold_idx, case_no, model_name, mode, m1)]
        b = main_attributions[(fold_idx, case_no, model_name, mode, m2)]
        rho, rank_dis, jaccard_dis = compare_importance(a, b, min(TOP_K, len(FEATURES)))
        within_rows.append({"fold": fold_idx, "case_no_within_fold": case_no, "mode": mode,
                            "model": model_name, "method_pair": f"{m1}__vs__{m2}",
                            "spearman_importance": rho, "rank_disagreement_0to1": rank_dis,
                            "topk_jaccard_disagreement": jaccard_dis})
within_model_disagreement = pd.DataFrame(within_rows)

# Across model types: compare the same explanation method on the same held-out case.
cross_rows = []
contexts2 = sorted({(k[0], k[1], k[3], k[4]) for k in main_attributions})
for fold_idx, case_no, mode, method in contexts2:
    models_here = sorted([k[2] for k in main_attributions if k[0] == fold_idx and k[1] == case_no and k[3] == mode and k[4] == method])
    for model_a, model_b in itertools.combinations(models_here, 2):
        a = main_attributions[(fold_idx, case_no, model_a, mode, method)]
        b = main_attributions[(fold_idx, case_no, model_b, mode, method)]
        rho, rank_dis, jaccard_dis = compare_importance(a, b, min(TOP_K, len(FEATURES)))
        cross_rows.append({"fold": fold_idx, "case_no_within_fold": case_no, "mode": mode,
                           "method": method, "model_pair": f"{model_a}__vs__{model_b}",
                           "spearman_importance": rho, "rank_disagreement_0to1": rank_dis,
                           "topk_jaccard_disagreement": jaccard_dis})
cross_model_disagreement = pd.DataFrame(cross_rows)

within_summary = (within_model_disagreement.groupby(["mode", "model", "method_pair"], dropna=False)
                  .agg(n_cases=("topk_jaccard_disagreement", "size"),
                       mean_topk_jaccard_disagreement=("topk_jaccard_disagreement", "mean"),
                       sd_topk_jaccard_disagreement=("topk_jaccard_disagreement", "std"),
                       mean_rank_disagreement=("rank_disagreement_0to1", "mean"),
                       mean_spearman=("spearman_importance", "mean"))
                  .reset_index())
cross_summary = (cross_model_disagreement.groupby(["mode", "method", "model_pair"], dropna=False)
                 .agg(n_cases=("topk_jaccard_disagreement", "size"),
                      mean_topk_jaccard_disagreement=("topk_jaccard_disagreement", "mean"),
                      sd_topk_jaccard_disagreement=("topk_jaccard_disagreement", "std"),
                      mean_rank_disagreement=("rank_disagreement_0to1", "mean"),
                      mean_spearman=("spearman_importance", "mean"))
                 .reset_index())

# Paired effects: positive value means M1 reduced disagreement compared with Baseline.
paired_rows = []
if not within_model_disagreement.empty:
    p = within_model_disagreement.pivot_table(
        index=["fold", "case_no_within_fold", "model", "method_pair"],
        columns="mode", values="topk_jaccard_disagreement", aggfunc="first").reset_index()
    if {"Baseline", "M1_shared_reference"}.issubset(p.columns):
        p["baseline_minus_m1_topk_disagreement"] = p["Baseline"] - p["M1_shared_reference"]
        p["improved_by_m1"] = p["baseline_minus_m1_topk_disagreement"] > 0
        paired_rows = p.to_dict("records")
paired_m1_effect = pd.DataFrame(paired_rows)

paired_cross_rows = []
if not cross_model_disagreement.empty:
    pc = cross_model_disagreement.pivot_table(
        index=["fold", "case_no_within_fold", "method", "model_pair"],
        columns="mode", values="topk_jaccard_disagreement", aggfunc="first").reset_index()
    if {"Baseline", "M1_shared_reference"}.issubset(pc.columns):
        pc["baseline_minus_m1_topk_disagreement"] = pc["Baseline"] - pc["M1_shared_reference"]
        paired_cross_rows = pc.to_dict("records")
paired_cross_m1_effect = pd.DataFrame(paired_cross_rows)

if not paired_m1_effect.empty:
    paired_m1_effect_summary = (paired_m1_effect.groupby(["model", "method_pair"], dropna=False)
        .agg(n_paired_cases=("baseline_minus_m1_topk_disagreement", "size"),
             mean_baseline_minus_m1=("baseline_minus_m1_topk_disagreement", "mean"),
             sd_baseline_minus_m1=("baseline_minus_m1_topk_disagreement", "std"),
             proportion_cases_improved=("improved_by_m1", "mean"))
        .reset_index())
else:
    paired_m1_effect_summary = pd.DataFrame()
if not paired_cross_m1_effect.empty:
    paired_cross_m1_effect_summary = (paired_cross_m1_effect.groupby(["method", "model_pair"], dropna=False)
        .agg(n_paired_cases=("baseline_minus_m1_topk_disagreement", "size"),
             mean_baseline_minus_m1=("baseline_minus_m1_topk_disagreement", "mean"),
             sd_baseline_minus_m1=("baseline_minus_m1_topk_disagreement", "std"))
        .reset_index())
else:
    paired_cross_m1_effect_summary = pd.DataFrame()

print("Within-model method disagreement:")
display(within_summary)
print("Across-model disagreement:")
display(cross_summary)
if not paired_m1_effect_summary.empty:
    print("Paired M1 effect summary (positive = lower disagreement after M1):")
    display(paired_m1_effect_summary)
if not paired_cross_m1_effect_summary.empty:
    print("Paired effect on cross-model disagreement:")
    display(paired_cross_m1_effect_summary)

## 7. Noise calibration
This estimates **within-explainer repeat/reference instability** on the first `NOISE_CASES_PER_FOLD` cases per fold. SHAP repeats use different Kernel SHAP sampling seeds while holding the background fixed; LIME repeats use different local perturbation seeds with a fixed training-reference bank; IG repeats resample its observed baseline/reference rows. Because these repeat mechanisms differ by explainer, report them as calibration estimates rather than as perfectly interchangeable noise measurements.

In [ ]:
noise_rows = []
for key, reps in noise_attributions.items():
    fold_idx, case_no, model_name, mode, method = key
    for i, j in itertools.combinations(range(len(reps)), 2):
        rho, rank_dis, jaccard_dis = compare_importance(reps[i], reps[j], min(TOP_K, len(FEATURES)))
        noise_rows.append({"fold": fold_idx, "case_no_within_fold": case_no, "model": model_name,
                           "mode": mode, "method": method, "repeat_i": i, "repeat_j": j,
                           "noise_scope": "IG_reference_resampling" if method == "IG" else "fixed_background_algorithmic_repeat",
                           "spearman_importance": rho, "rank_disagreement_0to1": rank_dis,
                           "topk_jaccard_disagreement": jaccard_dis})
noise_floor_detail = pd.DataFrame(noise_rows)
if not noise_floor_detail.empty:
    noise_floor_summary = (noise_floor_detail.groupby(["mode", "model", "method", "noise_scope"], dropna=False)
                           .agg(n_repeat_pairs=("topk_jaccard_disagreement", "size"),
                                mean_noise_topk_disagreement=("topk_jaccard_disagreement", "mean"),
                                sd_noise_topk_disagreement=("topk_jaccard_disagreement", "std"),
                                mean_noise_rank_disagreement=("rank_disagreement_0to1", "mean"))
                           .reset_index())
else:
    noise_floor_summary = pd.DataFrame()
print("Repeat/reference instability summary:")
display(noise_floor_summary)

# Descriptive comparison on the same 0-1 top-k Jaccard-disagreement scale.
# This is NOT a significance test: pairwise disagreement and single-method instability
# are different quantities, so read these side by side rather than treating this as correction.
noise_calibrated_rows = []
if not noise_floor_summary.empty and not within_summary.empty:
    for _, row in within_summary.iterrows():
        pair_methods = str(row["method_pair"]).split("__vs__")
        floors = noise_floor_summary[
            (noise_floor_summary["mode"] == row["mode"]) &
            (noise_floor_summary["model"] == row["model"]) &
            (noise_floor_summary["method"].isin(pair_methods))
        ]
        mean_floor = float(floors["mean_noise_topk_disagreement"].mean()) if len(floors) else np.nan
        noise_calibrated_rows.append({
            "mode": row["mode"], "model": row["model"], "method_pair": row["method_pair"],
            "mean_cross_explainer_disagreement": row["mean_topk_jaccard_disagreement"],
            "mean_pair_method_repeat_instability": mean_floor,
            "descriptive_excess_over_repeat_instability": row["mean_topk_jaccard_disagreement"] - mean_floor
                if np.isfinite(mean_floor) else np.nan,
            "interpretation_note": "descriptive diagnostic only; not a p-value or formal bias correction",
        })
noise_calibrated_summary = pd.DataFrame(noise_calibrated_rows)
print("Cross-explainer disagreement beside pair-method repeat instability (descriptive only):")
display(noise_calibrated_summary)


## 8. Realistic conditional feature-replacement faithfulness check
For each held-out case and feature, find an **observed training-fold donor** that is nearest on all *other* encoded features, then replace only the selected feature with that donor's observed value. This avoids zero-masking and avoids independently mixing multiple donors into one synthetic record. It is still an approximate conditional perturbation: a one-feature swap can never guarantee perfect joint plausibility, so the notebook records the distance to the nearest donor and you should inspect a small number of examples before writing claims.

The primary score is the Spearman correlation between explanation importance and the absolute change in predicted stunting probability after this one-feature conditional replacement. A top-k vs bottom-k probability-change gap is also reported. This measures faithfulness to the fitted model, not a causal relationship with stunting.

In [ ]:
faithfulness_detail_rows = []
faithfulness_instance_rows = []

for fold_idx, fold_ctx in fold_contexts.items():
    X_train = fold_ctx["X_train"].reset_index(drop=True)
    fitted_models = fold_ctx["fitted_models"]
    categorical = fold_ctx["categorical"]
    continuous = fold_ctx["continuous"]
    # Same feature transform per fold/model; transformed spaces have the same layout because types are fold-level.
    first_model = next(iter(fitted_models.values()))
    # Feature group mapping is model-specific but column ordering is equivalent across the three models.
    feature_groups = fold_ctx["feature_groups_by_model"]["LogisticRegression"]
    prep = first_model.named_steps["prep"]
    Z_train = np.asarray(prep.transform(X_train), dtype=float)
    for case_no in sorted(k[1] for k in case_context if k[0] == fold_idx):
        row = case_context[(fold_idx, case_no)]["row"][FEATURES].copy()
        Z_row = np.asarray(prep.transform(row.to_frame().T), dtype=float)[0]
        squared = (Z_train - Z_row.reshape(1, -1)) ** 2
        total_sq = squared.sum(axis=1)
        perturbed_rows = []
        donor_metadata = {}
        for f in FEATURES:
            cols = feature_groups[f]
            same_value = np.isclose(X_train[f].to_numpy(dtype=float), float(row[f]), equal_nan=True)
            candidate = np.where(~same_value)[0]
            if len(candidate) == 0:
                donor_metadata[f] = {"donor_value": float(row[f]), "donor_distance": np.nan, "available": False}
                perturbed_rows.append(row.copy())
                continue
            distance_other_sq = np.maximum(total_sq - squared[:, cols].sum(axis=1), 0.0)
            chosen = candidate[np.argmin(distance_other_sq[candidate])]
            donor_value = float(X_train.iloc[chosen][f])
            d = float(np.sqrt(distance_other_sq[chosen]))
            p_row = row.copy()
            p_row[f] = donor_value
            perturbed_rows.append(p_row)
            donor_metadata[f] = {"donor_value": donor_value, "donor_distance": d, "available": True}

        perturb_df = pd.DataFrame(perturbed_rows, columns=FEATURES)
        for model_name, model in fitted_models.items():
            p_original = float(model.predict_proba(row[FEATURES].to_frame().T)[:, 1][0])
            p_perturbed = model.predict_proba(perturb_df)[:, 1]
            delta_by_feature = np.abs(p_original - p_perturbed)
            for mode in ["Baseline", "M1_shared_reference"]:
                methods = ["SHAP", "LIME"] + (["IG"] if model_name == "MLP" else [])
                for method in methods:
                    key = (fold_idx, case_no, model_name, mode, method)
                    attr = main_attributions[key]
                    imp = np.abs(attr)
                    valid_idx = [i for i, f in enumerate(FEATURES) if donor_metadata[f]["available"]]
                    if len(valid_idx) >= 2:
                        rho = spearman_safe(imp[valid_idx], delta_by_feature[valid_idx])
                    else:
                        rho = np.nan
                    k = min(max(1, TOP_K), len(FEATURES))
                    top_idx = np.argsort(-imp)[:k]
                    bottom_idx = np.argsort(imp)[:k]
                    top_delta = float(np.mean(delta_by_feature[top_idx]))
                    bottom_delta = float(np.mean(delta_by_feature[bottom_idx]))
                    faithfulness_instance_rows.append({
                        "fold": fold_idx, "case_no_within_fold": case_no, "model": model_name,
                        "mode": mode, "method": method, "n_features_with_donor": len(valid_idx),
                        "importance_delta_spearman": rho, "topk_mean_probability_change": top_delta,
                        "bottomk_mean_probability_change": bottom_delta,
                        "topk_minus_bottomk_probability_change": top_delta - bottom_delta,
                    })
                    order = np.argsort(-imp)
                    rank_map = {int(idx): int(rank + 1) for rank, idx in enumerate(order)}
                    for i, f in enumerate(FEATURES):
                        meta = donor_metadata[f]
                        faithfulness_detail_rows.append({
                            "fold": fold_idx, "case_no_within_fold": case_no, "model": model_name,
                            "mode": mode, "method": method, "feature": f,
                            "importance_abs": float(imp[i]), "importance_rank": rank_map[i],
                            "p_original": p_original,
                            "p_after_conditional_replacement": float(p_perturbed[i]),
                            "absolute_probability_change": float(delta_by_feature[i]),
                            "donor_distance_on_other_features": meta["donor_distance"],
                            "donor_available": bool(meta["available"]),
                        })

faithfulness_detail = pd.DataFrame(faithfulness_detail_rows)
faithfulness_instance = pd.DataFrame(faithfulness_instance_rows)
faithfulness_summary = (faithfulness_instance.groupby(["mode", "model", "method"], dropna=False)
                        .agg(n_case_checks=("importance_delta_spearman", "size"),
                             mean_importance_delta_spearman=("importance_delta_spearman", "mean"),
                             sd_importance_delta_spearman=("importance_delta_spearman", "std"),
                             mean_topk_probability_change=("topk_mean_probability_change", "mean"),
                             mean_bottomk_probability_change=("bottomk_mean_probability_change", "mean"),
                             mean_topk_minus_bottomk_change=("topk_minus_bottomk_probability_change", "mean"))
                        .reset_index())
print("Conditional replacement faithfulness summary:")
display(faithfulness_summary)

## 9. Save results and quick visual checks
All output files exclude raw child/household IDs. Results are saved in a timestamped run folder under `Thesis/results/` when Drive is mounted; otherwise they are saved under `/content/m1_pipeline_results/`.

In [ ]:
run_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
try:
    root = Path(RESULTS_ROOT)
    root.mkdir(parents=True, exist_ok=True)
    output_dir = root / f"m1_shared_reference_{FEATURE_SET_LABEL}_{RUN_MODE}_{run_stamp}"
    output_dir.mkdir(parents=True, exist_ok=True)
except Exception:
    output_dir = Path("/content/m1_pipeline_results") / f"m1_shared_reference_{FEATURE_SET_LABEL}_{RUN_MODE}_{run_stamp}"
    output_dir.mkdir(parents=True, exist_ok=True)

outputs = {
    "fold_metrics.csv": fold_metrics,
    "explanations_long.csv": explanations_long,
    "within_model_method_disagreement.csv": within_model_disagreement,
    "within_model_disagreement_summary.csv": within_summary,
    "cross_model_disagreement.csv": cross_model_disagreement,
    "cross_model_disagreement_summary.csv": cross_summary,
    "paired_m1_effect_within_model.csv": paired_m1_effect,
    "paired_m1_effect_within_model_summary.csv": paired_m1_effect_summary,
    "paired_m1_effect_across_models.csv": paired_cross_m1_effect,
    "paired_m1_effect_across_models_summary.csv": paired_cross_m1_effect_summary,
    "noise_floor_detail.csv": noise_floor_detail,
    "noise_floor_summary.csv": noise_floor_summary,
    "noise_calibrated_comparison.csv": noise_calibrated_summary,
    "faithfulness_detail.csv": faithfulness_detail,
    "faithfulness_instance_summary.csv": faithfulness_instance,
    "faithfulness_summary.csv": faithfulness_summary,
}
for filename, frame in outputs.items():
    frame.to_csv(output_dir / filename, index=False)

metadata = {
    "title": "M1 Shared-Reference Mitigation Pipeline — BDHS 2022 Child Stunting",
    "run_timestamp": run_stamp,
    "data_file_basename": Path(DATA_PATH_RESOLVED).name,
    "feature_set_mode": FEATURE_SET_MODE,
    "feature_set_label": FEATURE_SET_LABEL,
    "features": FEATURES,
    "target_definition": f"{TARGET_COL} < {STUNTING_CUTOFF}; valid target range [{TARGET_VALID_MIN},{TARGET_VALID_MAX}]",
    "cluster_column": CLUSTER_COL,
    "cv": {"type": "StratifiedGroupKFold", "n_splits": N_SPLITS, "seed": SEED},
    "run_mode": RUN_MODE,
    "explained_cases_per_fold": N_EXPLAIN_PER_FOLD,
    "reference_rows_per_fold": min(N_REFERENCE_ROWS, len(analytic)),
    "shap_nsamples": SHAP_NSAMPLES,
    "lime_num_samples": LIME_NUM_SAMPLES,
    "ig_steps": IG_STEPS,
    "noise_cases_per_fold": NOISE_CASES_PER_FOLD,
    "noise_repeats_total": NOISE_REPEATS_TOTAL,
    "models": ["LogisticRegression", "RandomForest", "MLP"],
    "explainers": {"LogisticRegression": ["SHAP", "LIME"], "RandomForest": ["SHAP", "LIME"], "MLP": ["SHAP", "LIME", "IG"]},
    "m1_definition": "same observed training-reference rows used by SHAP, LIME training data, and MLP IG baselines within each fold",
    "faithfulness_definition": "one feature at a time replaced by an observed training donor nearest on all other transformed features",
    "stunting_prevalence_unweighted_analytic": float(np.mean(y)),
    "analytic_n": int(len(analytic)),
    "analytic_clusters": int(pd.Series(groups).nunique()),
    "warnings": [
        "DEBUG_4 is not the Arm A 14-feature thesis configuration.",
        "Explanations are associations with model outputs, not causal effects.",
        "LIME's internal local perturbations are method-generated; the faithfulness check uses observed donor values.",
        "Noise estimates differ in what is repeated by method; interpret them separately.",
    ],
}
with open(output_dir / "run_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

# Main paired disagreement plot for within-model explainer comparisons.
if not within_model_disagreement.empty:
    plot_df = within_model_disagreement.groupby(["mode", "model", "method_pair"])["topk_jaccard_disagreement"].mean().reset_index()
    pivot = plot_df.pivot_table(index=["model", "method_pair"], columns="mode", values="topk_jaccard_disagreement")
    ax = pivot.plot(kind="bar", figsize=(12, 5))
    ax.set_ylabel("Mean top-k Jaccard disagreement (lower is better)")
    ax.set_title("Baseline vs M1: within-model explanation disagreement")
    ax.set_ylim(0, 1)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(output_dir / "baseline_vs_m1_disagreement.png", dpi=160)
    plt.show()

if not faithfulness_summary.empty:
    plot_df = faithfulness_summary.copy()
    plot_df["label"] = plot_df["model"] + " / " + plot_df["method"] + " / " + plot_df["mode"]
    ax = plot_df.plot(x="label", y="mean_importance_delta_spearman", kind="bar", figsize=(13, 5), legend=False)
    ax.set_ylabel("Mean Spearman(importance, probability change)")
    ax.set_title("Conditional replacement faithfulness (higher is better)")
    plt.xticks(rotation=60, ha="right")
    plt.tight_layout()
    plt.savefig(output_dir / "conditional_replacement_faithfulness.png", dpi=160)
    plt.show()

print("\nSaved results to:", output_dir)
print("Files:")
for p in sorted(output_dir.iterdir()):
    print(" -", p.name)

## 10. How to interpret the results
- **M1 works on disagreement** only if the relevant Baseline-to-M1 disagreement metrics decrease across held-out cases/folds. `baseline_minus_m1_topk_disagreement > 0` means lower disagreement after M1.
- **Noise calibration:** compare cross-explainer disagreement with within-explainer repeat/reference instability, but keep the repeat mechanism in view. SHAP/LIME repeats are primarily algorithmic; IG repeats resample baseline rows.
- **Faithfulness:** positive `topk_minus_bottomk_probability_change` and positive importance-vs-change correlation are evidence that highly ranked features more strongly affect the fitted model under this particular conditional-replacement check. They are not proof of causal importance.
- **Predictive quality:** AUROC/AP/Brier are reported to contextualize explanations. Good predictive performance does not automatically imply good explanations.
- Do not report the default `DEBUG_4` run as the thesis Arm A result. The results metadata marks it explicitly as a smoke test. After it works, update the exact 14 Arm A features and rerun with `RUN_MODE = "full"`.
- For a paper, add fold-level uncertainty intervals and a predeclared primary disagreement metric. Avoid treating all child-level rows as independent because observations are clustered by survey PSU.